## Line Overlay Inspector (ipywidgets)\n
\n
Interactive viewer for frame images with per-label polyline overlays.

In [1]:
import ipywidgets as W
from IPython.display import display


In [2]:
from pathlib import Path

from line_overlay_utils import (
    DatasetPaths,
    list_frames_in_images_dir,
    list_label_types,
    load_polyline_csv,
    render_overlay_matplotlib,
)

DATASET_DIR = r"C:\\Users\\wanglab\\Desktop\\Mel\\Mel_MRN_opto_cohort_videos\\TRAINING_DATA\\MRN_opto5_F_8_2_25_L_20260116_opto_xp_2s_dur_10x_stimulations0_nob"

paths = DatasetPaths(dataset_dir=Path(DATASET_DIR))
frames = list_frames_in_images_dir(paths.images_dir, image_prefix=paths.image_prefix, image_ext=paths.image_ext)
label_types = list_label_types(paths.labels_dir)

print(f"Images dir: {paths.images_dir}")
print(f"Labels dir: {paths.labels_dir}")
print(f"Frames found: {len(frames)}")
print(f"Label folders: {label_types}")
if frames:
    print(f"Frame range: {min(frames)} to {max(frames)}")


Images dir: C:\Users\wanglab\Desktop\Mel\Mel_MRN_opto_cohort_videos\TRAINING_DATA\MRN_opto5_F_8_2_25_L_20260116_opto_xp_2s_dur_10x_stimulations0_nob\images
Labels dir: C:\Users\wanglab\Desktop\Mel\Mel_MRN_opto_cohort_videos\TRAINING_DATA\MRN_opto5_F_8_2_25_L_20260116_opto_xp_2s_dur_10x_stimulations0_nob\labels
Frames found: 686
Label folders: ['0', '1', '2', '3', '4', '5']
Frame range: 48505 to 53024


In [ ]:
from IPython.display import clear_output

dataset_text = W.Text(
    value=DATASET_DIR,
    description='dataset',
    layout=W.Layout(width='900px'),
)
reload_btn = W.Button(description='reload')

label_dropdown = W.Dropdown(
    options=label_types if label_types else ['0'],
    value=(label_types[0] if label_types else '0'),
    description='label',
    layout=W.Layout(width='220px'),
)

# Restrict selection to existing frame IDs only.
_frame_options = frames[:] if frames else [0]
frame_slider = W.SelectionSlider(
    options=_frame_options,
    value=_frame_options[0],
    description='frame',
    continuous_update=False,
    layout=W.Layout(width='650px'),
)
frame_text = W.BoundedIntText(
    value=_frame_options[0],
    min=min(_frame_options),
    max=max(_frame_options),
    description='frame',
    layout=W.Layout(width='220px'),
)

show_points = W.Checkbox(value=False, description='show points')
status = W.HTML(value='')
out = W.Output()


def _load_state(dataset_dir: str):
    p = DatasetPaths(dataset_dir=Path(dataset_dir))
    fr = list_frames_in_images_dir(p.images_dir, image_prefix=p.image_prefix, image_ext=p.image_ext)
    lt = list_label_types(p.labels_dir)
    return p, fr, lt


def _sync_text_from_slider(*_):
    frame_text.value = int(frame_slider.value)


def _sync_slider_from_text(*_):
    available = list(frame_slider.options)
    if not available:
        return
    requested = int(frame_text.value)
    if requested in available:
        frame_slider.value = requested
    else:
        nearest = min(available, key=lambda x: abs(x - requested))
        frame_slider.value = nearest
        frame_text.value = nearest


def _render(*_):
    with out:
        clear_output(wait=True)
        frame = int(frame_slider.value)
        label_type = str(label_dropdown.value)
        img_path = paths.image_path(frame)
        csv_path = paths.label_csv_path(label_type, frame)

        pts = load_polyline_csv(csv_path)
        if pts.shape[0] == 0:
            print(f'No label found for label={label_type}, frame={frame} (expected: {csv_path})')

        render_overlay_matplotlib(
            img_path,
            pts,
            alpha=0.5,
            show_points=bool(show_points.value),
        )


def _reload(*_):
    global paths, frames
    paths, frames, lts = _load_state(dataset_text.value)

    available_frames = frames[:] if frames else [0]
    current = int(frame_slider.value) if getattr(frame_slider, 'value', None) is not None else available_frames[0]

    frame_slider.options = available_frames
    frame_slider.value = current if current in available_frames else available_frames[0]

    frame_text.min = min(available_frames)
    frame_text.max = max(available_frames)
    frame_text.value = int(frame_slider.value)

    if lts:
        label_dropdown.options = lts
        if label_dropdown.value not in lts:
            label_dropdown.value = lts[0]
    else:
        label_dropdown.options = ['0']
        label_dropdown.value = '0'

    status.value = (
        f"<pre>Images: {paths.images_dir}\nLabels: {paths.labels_dir}\nFrames: {len(frames)}\nLabel types: {list(label_dropdown.options)}</pre>"
    )
    _render()


reload_btn.on_click(_reload)
frame_slider.observe(_sync_text_from_slider, names='value')
frame_slider.observe(_render, names='value')
frame_text.observe(_sync_slider_from_text, names='value')
label_dropdown.observe(_render, names='value')
show_points.observe(_render, names='value')

controls = W.VBox([
    W.HBox([dataset_text, reload_btn]),
    status,
    W.HBox([label_dropdown, show_points]),
    W.HBox([frame_slider, frame_text]),
])

display(controls, out)
_reload()


Output()